In [15]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
import evaluate

In [16]:
#data prep

odf = pd.read_csv("w22_entropy_differences_full.csv")
fdf = odf.copy()
fdf = fdf.drop_duplicates(subset=["sentence_id"])
fdf = fdf[["text", "label_binary"]]  #.sample(n=50, random_state=42)

#dropping followed news sources cuz we can deal with later


#small for testing
# samped_sentences = fdf["sentence_id"].sample(n=2, random_state=42)
# fdf = fdf[fdf["sentence_id"].isin(samped_sentences)]  #

In [17]:


# ==== CONFIG ====
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # or whichever TinyLlama ckpt you want
MAX_LENGTH = 128


# ==== DATASET PREP ====
fdf = fdf.rename(columns={"label_binary": "labels"})

# Turn pandas -> HF Dataset and split
dataset = Dataset.from_pandas(fdf, preserve_index=False)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_ds = dataset["train"]
eval_ds = dataset["test"]

# ==== TOKENIZER ====
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Many LLaMA-style models lack a pad token; use eos as pad
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )





In [18]:

train_ds = train_ds.map(tokenize_fn, batched=True)
eval_ds = eval_ds.map(tokenize_fn, batched=True)

# We don't need the raw text in the model inputs
train_ds = train_ds.remove_columns(["text"])
eval_ds = eval_ds.remove_columns(["text"])

# Set format for PyTorch
train_ds.set_format(type="torch")
eval_ds.set_format(type="torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


# ==== MODEL ====
class TinyLlamaClassifier(nn.Module):
    def __init__(self, base_model_name, num_labels=1):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.base_model.config.hidden_size

        # Simple linear layer to one logit (binary classification)
        self.classifier = nn.Linear(hidden_size, num_labels)

        # Freeze base model
        for param in self.base_model.parameters():
            param.requires_grad = False

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        # outputs.last_hidden_state: [batch, seq_len, hidden]
        last_hidden = outputs.last_hidden_state

        # Mean pooling over non-padded tokens
        mask = attention_mask.unsqueeze(-1)  # [batch, seq_len, 1]
        masked_hidden = last_hidden * mask
        sum_hidden = masked_hidden.sum(dim=1)  # [batch, hidden]
        lengths = mask.sum(dim=1)  # [batch, 1]
        pooled = sum_hidden / lengths.clamp(min=1.0)

        logits = self.classifier(pooled).squeeze(-1)  # [batch]

        loss = None
        if labels is not None:
            labels = labels.float()
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}


model = TinyLlamaClassifier(BASE_MODEL)

Map:   0%|          | 0/1360 [00:00<?, ? examples/s]

Map:   0%|          | 0/340 [00:00<?, ? examples/s]

In [19]:

# ==== METRICS ====
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs > 0.5).astype(int)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]
    return {"accuracy": acc, "f1": f1}


# ==== TRAINER ====
training_args = TrainingArguments(
    output_dir="./tinyllama_binary_cls",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=5e-4,   # only the small head is trainable, so can be a bit higher
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),   # if you're on GPU with fp16 support
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ==== TRAIN ====
trainer.train()

# # ==== (OPTIONAL) SAVE ====
# trainer.save_model("./tinyllama_binary_cls/best_model")
# tokenizer.save_pretrained("./tinyllama_binary_cls/best_model")

/scratch/slurm-1496098/ipykernel_2733596/2914672108.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.634200,0.644521,0.679412,0.773389
2,0.595700,0.646021,0.658824,0.752137
3,0.529200,0.646479,0.658824,0.753191


TrainOutput(global_step=510, training_loss=0.5950023071438658, metrics={'train_runtime': 96.0834, 'train_samples_per_second': 42.463, 'train_steps_per_second': 5.308, 'total_flos': 0.0, 'train_loss': 0.5950023071438658, 'epoch': 3.0})

In [20]:
eval_metrics = trainer.evaluate(eval_dataset=eval_ds)
print("Final eval metrics:", eval_metrics)

Final eval metrics: {'eval_loss': 0.6445207595825195, 'eval_accuracy': 0.6794117647058824, 'eval_f1': 0.7733887733887734, 'eval_runtime': 1.358, 'eval_samples_per_second': 250.37, 'eval_steps_per_second': 31.664, 'epoch': 3.0}


In [21]:
import pandas as pd
from datasets import Dataset
import numpy as np

fdf_full = odf.copy()
fdf_full = fdf_full.drop_duplicates(subset=["sentence_id"])
fdf_full = fdf_full[["text", "label_binary"]]

# HF likes the column name "labels" but we don't actually *need* labels for prediction;
# still, we can keep them for convenience.
fdf_full_hf = fdf_full.rename(columns={"label_binary": "labels"})

full_dataset = Dataset.from_pandas(fdf_full_hf, preserve_index=False)

# Reuse the same tokenize_fn you used for train/eval
full_dataset = full_dataset.map(tokenize_fn, batched=True)

# Remove raw text so only tensors remain
full_dataset = full_dataset.remove_columns(["text"])

full_dataset.set_format(type="torch")

# ==== PREDICT ON FULL DATASET ====
pred_output = trainer.predict(full_dataset)
logits = pred_output.predictions  # shape: (N,)

# Convert logits -> probabilities -> binary predictions
probs = 1 / (1 + np.exp(-logits))
preds = (probs > 0.5).astype(int)

# Attach predictions back to a DataFrame
fdf_full["pred_prob"] = probs
fdf_full["pred_label"] = preds

# Optionally: keep original label for comparison, compute quick sanity accuracy
acc_full = (fdf_full["pred_label"] == fdf_full["label_binary"]).mean()
print(f"Accuracy on full dataset (using ground truth labels): {acc_full:.4f}")

# ==== SAVE TO DISK ====
fdf_full.to_csv("tinyllama_full_predictions.csv", index=False)
print("Saved predictions to tinyllama_full_predictions.csv")


Map:   0%|          | 0/1700 [00:00<?, ? examples/s]

Accuracy on full dataset (using ground truth labels): 0.7000
Saved predictions to tinyllama_full_predictions.csv


In [22]:
eval_metrics = trainer.evaluate(eval_dataset=full_dataset)
print("Final eval metrics:", eval_metrics)

Final eval metrics: {'eval_loss': 0.5907338857650757, 'eval_accuracy': 0.7, 'eval_f1': 0.7857142857142857, 'eval_runtime': 6.3181, 'eval_samples_per_second': 269.07, 'eval_steps_per_second': 33.713, 'epoch': 3.0}
